In [4]:
import json
import pandas as pd

# FILES

INPUT_JSON = "gov_data_be.json"

SUSPICIOUS_KEYWORDS_FILE = (
    "suspicious_keywords.csv"
)

OUTPUT_JSON = (
    "gov_data_be_clean.json"
)

# LOAD DATA

print("\nLoading datasets...")

with open(
    INPUT_JSON,
    "r",
    encoding="utf-8"
) as f:

    datasets = json.load(f)

print(
    f"Datasets loaded: "
    f"{len(datasets)}"
)

# LOAD SUSPICIOUS KEYWORDS

print(
    "\nLoading suspicious keywords..."
)

df_suspicious = pd.read_csv(
    SUSPICIOUS_KEYWORDS_FILE
)

remove_keywords = set(

    df_suspicious[
        "keyword"
    ]
    .astype(str)
    .str.strip()
)

print(
    f"Keywords flagged for removal: "
    f"{len(remove_keywords)}"
)

# TEXT NORMALIZATION

NORMALIZATION_MAP = {

    # Description fragments

    "Meer informatie":
        "More information",

    "Variabeles":
        "Variables",

    "Periode":
        "Period",

    # Keywords

    "infectie":
        "infection",

    "HIV-infectie":
        "HIV infection",

    "HIV-infection":
        "HIV infection",

    "alcoholische handontsmetting":
        "alcoholic hand sanitizer"
}

def normalize_text(text):

    if not isinstance(text, str):
        return text

    for old, new in NORMALIZATION_MAP.items():

        text = text.replace(
            old,
            new
        )

    return text

# CLEAN DATASETS

clean_datasets = []

datasets_removed = 0

keywords_before = 0
keywords_removed = 0
keywords_after = 0

titles_normalized = 0
descriptions_normalized = 0
keywords_normalized = 0

duplicates_removed = 0

for dataset in datasets:

    # NORMALIZE TITLE

    if "title" in dataset:

        original_title = dataset["title"]

        dataset["title"] = normalize_text(
            dataset["title"]
        )

        if dataset["title"] != original_title:

            titles_normalized += 1

    # NORMALIZE DESCRIPTION

    if "description" in dataset:

        original_description = (
            dataset["description"]
        )

        dataset["description"] = (
            normalize_text(
                dataset["description"]
            )
        )

        if (
            dataset["description"]
            != original_description
        ):

            descriptions_normalized += 1

    # KEYWORDS

    original_keywords = (
        dataset.get(
            "keywords",
            []
        )
    )

    keywords_before += (
        len(original_keywords)
    )

    cleaned_keywords = []

    for kw in original_keywords:

        kw_clean = kw.strip()

        # REMOVE SUSPICIOUS KEYWORDS

        if kw_clean in remove_keywords:

            keywords_removed += 1

            continue

        # NORMALIZE KEYWORDS

        normalized_kw = (
            normalize_text(
                kw_clean
            )
        )

        if normalized_kw != kw_clean:

            keywords_normalized += 1

        cleaned_keywords.append(
            normalized_kw
        )

    # DEDUPLICATE KEYWORDS

    before_dedup = len(
        cleaned_keywords
    )

    cleaned_keywords = sorted(
        list(
            set(
                cleaned_keywords
            )
        )
    )

    duplicates_removed += (
        before_dedup
        - len(cleaned_keywords)
    )

    keywords_after += (
        len(cleaned_keywords)
    )

    # REMOVE DATASETS WITH NO KEYWORDS

    if len(cleaned_keywords) == 0:

        datasets_removed += 1

        continue

    dataset["keywords"] = (
        cleaned_keywords
    )

    clean_datasets.append(
        dataset
    )

# SAVE CLEAN FILE

with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        clean_datasets,
        f,
        indent=2,
        ensure_ascii=False
    )

# STATISTICS

print("\n")
print("=" * 70)
print("CLEANING SUMMARY")
print("=" * 70)

print(
    f"Original datasets: "
    f"{len(datasets)}"
)

print(
    f"Datasets removed: "
    f"{datasets_removed}"
)

print(
    f"Final datasets: "
    f"{len(clean_datasets)}"
)

print()

print(
    f"Keywords before cleaning: "
    f"{keywords_before}"
)

print(
    f"Keywords removed: "
    f"{keywords_removed}"
)

print(
    f"Keywords normalized: "
    f"{keywords_normalized}"
)

print(
    f"Duplicate keywords removed: "
    f"{duplicates_removed}"
)

print(
    f"Keywords after cleaning: "
    f"{keywords_after}"
)

if keywords_before > 0:

    print(
        f"Percent keywords removed: "
        f"{(keywords_removed / keywords_before) * 100:.2f}%"
    )

print()

print(
    f"Titles normalized: "
    f"{titles_normalized}"
)

print(
    f"Descriptions normalized: "
    f"{descriptions_normalized}"
)

print()

print(
    f"Output file: "
    f"{OUTPUT_JSON}"
)

print("=" * 70)


Loading datasets...
Datasets loaded: 2255

Loading suspicious keywords...
Keywords flagged for removal: 24


CLEANING SUMMARY
Original datasets: 2255
Datasets removed: 1
Final datasets: 2254

Keywords before cleaning: 10019
Keywords removed: 87
Keywords normalized: 58
Duplicate keywords removed: 58
Keywords after cleaning: 9874
Percent keywords removed: 0.87%

Titles normalized: 0
Descriptions normalized: 20

Output file: gov_data_be_clean.json
